# 03 — Detección Temporal de Jugadas, Rotación de Slot e Identificación de Carta

Este notebook implementa la primera parte del **Paso 4B** del pipeline:
1. **Consume únicamente las regiones confiables de los manifiestos en Cloudflare R2:**
   - `metadata/segmented_matches.json` (Notebook 01: `start_sec`, `end_sec`, `start_clock`, `end_clock`).
   - `metadata/ui_layouts.json` (Notebook 02: **únicamente `elixir_bar_box` y los 4 `card_slots`**, sin usar `next_card_box` ni `emote_button_box`).
   - `metadata/cards_catalog.json` + iconos `assets/cards/<key>.png` (Notebook 00: 121 cartas base + 39 evoluciones).
2. **Sincronización Exacta del Reloj de Batalla (`start_clock`) y Fases de Elixir (1x, 2x, 3x):**
   - Calcula el desfase inicial $\text{offset} = 180 - \text{parse}(\text{start\_clock})$ para alinear al segundo exacto los cambios a **2x** ($120\text{ s}$) y **3x** ($240\text{ s}$).
   - Muestrea secuencialmente a **5 FPS ($\Delta t = 0.20\text{ s}$)** y compensa la regeneración natural ($2.8\text{ s}$, $1.4\text{ s}$ o $0.7\text{ s}$ por gota).
3. **Restricción Física de Elixir + Inferencia del Mazo de 8 Cartas:**
   - Descarta cartas cuyo coste de elixir supere el elixir disponible (`elixir_before`) o difiera drásticamente del gasto medido.
   - Infiere automáticamente las **8 cartas del mazo del jugador** en cada partida y refina las detecciones dudosas dentro de ese mazo.
4. **Generación de Videos de Debug sin Reprocesar (Última Celda):**
   - Lee directamente los checkpoints ya guardados en R2 (`metadata/events/<video_id>.json`). Puedes generar videos de contraste para 2 videos hoy y más adelante generarlos para cualquier otro video o para todas sus partidas sin volver a correr la extracción.

In [61]:
import os, sys, json, time, subprocess, base64
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import requests
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import boto3
from botocore.config import Config

# 1. Directorios de trabajo
WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('./working')
VIDEOS_DIR = WORK_DIR / 'videos'
CARDS_DIR = WORK_DIR / 'cards_cache'
EVENTS_DIR = WORK_DIR / 'events'
MANIFESTS_DIR = WORK_DIR / 'manifests'
DEBUG_VIDEOS_DIR = WORK_DIR / 'debug_videos'

for d in [VIDEOS_DIR, CARDS_DIR, EVENTS_DIR, MANIFESTS_DIR, DEBUG_VIDEOS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 2. Carga de credenciales (.env local o Kaggle Secrets)
for env_candidate in [Path('.env'), Path('../.env')]:
    if env_candidate.exists():
        for line in env_candidate.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

def get_config(key: str, default: str = None) -> str:
    if os.getenv(key):
        return os.getenv(key)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        return default

R2_ENDPOINT = get_config('R2_ENDPOINT')
R2_ACCESS_KEY = get_config('R2_ACCESS_KEY')
R2_SECRET_KEY = get_config('R2_SECRET_KEY')
R2_BUCKET = get_config('R2_BUCKET', 'cr-videos')
CDN_BASE = get_config('CDN_URL', 'https://cdn-cr.eddux.dev').rstrip('/')

s3_client = boto3.client(
    's3',
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
    config=Config(signature_version='s3v4', max_pool_connections=32)
)
print(f"✔ Conectado a Cloudflare R2 (Bucket: '{R2_BUCKET}') | CDN: {CDN_BASE}")

✔ Conectado a Cloudflare R2 (Bucket: 'cr-videos') | CDN: https://cdn-cr.eddux.dev


In [62]:
# =========================================================================
# CARGA DE MANIFIESTOS DESDE R2 Y FILTRO DE SELECCIÓN DE VIDEOS
# =========================================================================

def load_r2_json(r2_key: str, local_fallback: Path) -> dict:
    try:
        obj = s3_client.get_object(Bucket=R2_BUCKET, Key=r2_key)
        data = json.loads(obj['Body'].read().decode('utf-8'))
        local_fallback.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
        return data
    except Exception as e:
        print(f'⚠ Error leyendo {r2_key} desde R2 ({e}), intentando copia local...')
        if local_fallback.exists():
            return json.loads(local_fallback.read_text(encoding='utf-8'))
        raise

segmented_manifest = load_r2_json('metadata/segmented_matches.json', MANIFESTS_DIR / 'segmented_matches.json')
layouts_manifest   = load_r2_json('metadata/ui_layouts.json', MANIFESTS_DIR / 'ui_layouts.json')
cards_manifest     = load_r2_json('metadata/cards_catalog.json', MANIFESTS_DIR / 'cards_catalog.json')

videos_seg_dict = segmented_manifest.get('videos', {})
layouts_dict    = layouts_manifest.get('layouts', {})
cards_dict      = cards_manifest.get('cards', {})

available_videos = []
for vid, seg_info in videos_seg_dict.items():
    matches = seg_info.get('matches', [])
    if matches and vid in layouts_dict:
        available_videos.append({
            'id': vid,
            'title': seg_info.get('title', vid),
            'matches': matches,
            'layout': layouts_dict[vid],
            'total_duration_s': sum(m.get('duration_sec', 0) for m in matches)
        })

available_videos.sort(key=lambda x: (0 if x['total_duration_s'] >= 120 else 1, len(x['matches']), x['total_duration_s']))

def get_processed_event_ids() -> set[str]:
    done = {f.stem for f in EVENTS_DIR.glob('*.json')}
    try:
        res = s3_client.list_objects_v2(Bucket=R2_BUCKET, Prefix='metadata/events/')
        for item in res.get('Contents', []):
            done.add(Path(item['Key']).stem)
    except Exception:
        pass
    return done

already_extracted = get_processed_event_ids()

# =========================================================================
# SELECTOR DE VIDEOS PARA EXTRACCIÓN DE JUGADAS
# - Entero (ej. 1): prueba rápida con 1 video
# - String (ej. 'Qy18H5V-nbE'): un video específico por ID
# - Lista (ej. ['Qy18H5V-nbE', '2Q9wFQfkVu4']): varios IDs
# - 'all': procesar todos los videos del dataset
# =========================================================================
VIDEO_FILTER = 'all'

def select_videos(calibrations_list, filter_rule):
    if filter_rule == 'all':
        return calibrations_list
    if isinstance(filter_rule, int):
        return calibrations_list[:filter_rule]
    if isinstance(filter_rule, str):
        return [c for c in calibrations_list if c['id'] == filter_rule]
    if isinstance(filter_rule, (list, tuple, set)):
        return [c for c in calibrations_list if c['id'] in filter_rule]
    return calibrations_list

selected_videos = select_videos(available_videos, VIDEO_FILTER)
print(f'✔ Videos con partidas y layout listos: {len(available_videos)} | Ya procesados en R2: {len(already_extracted)}')
print(f'✔ Seleccionados con VIDEO_FILTER={VIDEO_FILTER!r}: {len(selected_videos)}')
for v in selected_videos[:5]:
    m0 = v['matches'][0]
    print(f"  • {v['id']} ({len(v['matches'])} partida(s), {v['total_duration_s']}s, start_clock={m0.get('start_clock')}) — {v['title'][:45]}")

✔ Videos con partidas y layout listos: 41 | Ya procesados en R2: 33
✔ Seleccionados con VIDEO_FILTER='all': 41
  • FsfSvTL6kIs (1 partida(s), 152s, start_clock=2:57) — Clash Royale partida sin comentarios - Arena 
  • XTiXwJ8ffRM (1 partida(s), 153s, start_clock=2:55) — Clash Royale partida sin comentarios - Arena 
  • qlB3ewZywzs (1 partida(s), 160s, start_clock=2:56) — Clash Royale partida sin comentarios - arena 
  • AMMLa-tcCNM (1 partida(s), 164s, start_clock=2:56) — Clash Royale partida sin comentarios - arena 
  • 4YR1phwfrJU (1 partida(s), 167s, start_clock=2:48) — ✨👀Partida de #clashroyale sin comentarios #Cl


In [63]:
# =========================================================================
# BANCO DE PLANTILLAS DE CARTAS (121 BASE + 39 EVOLUCIONES)
# =========================================================================

TEMPLATE_W, TEMPLATE_H = 52, 60

def _prepare_card_template(card_entry: dict) -> dict:
    key = card_entry['key']
    local_png = CARDS_DIR / f'{key}.png'
    if not local_png.exists():
        r = requests.get(card_entry['cdn_url'], timeout=15)
        r.raise_for_status()
        local_png.write_bytes(r.content)

    pil_rgba = Image.open(local_png).convert('RGBA')
    bg = Image.new('RGBA', pil_rgba.size, (32, 54, 86, 255))
    comp_rgb = np.array(Image.alpha_composite(bg, pil_rgba).convert('RGB'))
    th, tw = comp_rgb.shape[:2]

    art_crop = comp_rgb[int(th * 0.16):int(th * 0.78), int(tw * 0.12):int(tw * 0.88)]
    art_resized = cv2.resize(art_crop, (TEMPLATE_W, TEMPLATE_H), interpolation=cv2.INTER_AREA)
    art_lab = cv2.cvtColor(art_resized, cv2.COLOR_RGB2LAB).astype(np.float32)
    art_gray = cv2.cvtColor(art_resized, cv2.COLOR_RGB2GRAY).astype(np.float32)

    art_hsv = cv2.cvtColor(art_resized, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([art_hsv], [0, 1], None, [18, 16], [0, 180, 0, 256])
    cv2.normalize(hist, hist, alpha=1.0, norm_type=cv2.NORM_L1)

    return {
        **card_entry,
        'full_rgb': comp_rgb,
        'art_rgb': art_resized,
        'art_lab': art_lab,
        'art_gray': art_gray,
        'hsv_hist': hist
    }

standard_cards = [c for c in cards_dict.values() if c.get('is_standard_deck_card')]
t0 = time.time()
with ThreadPoolExecutor(max_workers=24) as pool:
    card_templates = list(pool.map(_prepare_card_template, standard_cards))

card_templates_by_key = {c['key']: c for c in card_templates}
print(f'✔ Banco de {len(card_templates)} plantillas de cartas preparado en {time.time() - t0:.2f}s.')

✔ Banco de 160 plantillas de cartas preparado en 0.29s.


In [64]:
# =========================================================================
# ALGORITMOS CORE: RELOJ INICIAL (start_clock), BARRA DE ELIXIR,
# ROTACIÓN DE SLOT E IDENTIFICACIÓN CON RESTRICCIÓN FÍSICA Y MAZO DE 8 CARTAS
# =========================================================================

def fmt_mmss(seconds: float) -> str:
    m, s = divmod(int(round(seconds)), 60)
    return f'{m}:{s:02d}'

def parse_clock_to_seconds(clock_str: str, default_sec: int = 176) -> int:
    if not clock_str or not isinstance(clock_str, str) or ':' not in clock_str:
        return default_sec
    try:
        m_str, s_str = clock_str.strip().split(':', 1)
        val = int(m_str) * 60 + int(s_str)
        return val if 120 <= val <= 180 else default_sec
    except Exception:
        return default_sec

def get_battle_clock_offset(match_info: dict) -> float:
    start_clock_sec = parse_clock_to_seconds(match_info.get('start_clock', '2:56'))
    return float(max(0, 180 - start_clock_sec))

def get_elixir_regen_rate(battle_elapsed_sec: float) -> tuple[float, str]:
    if battle_elapsed_sec < 120.0:
        return 1.0 / 2.8, '1x'
    elif battle_elapsed_sec < 240.0:
        return 1.0 / 1.4, '2x'
    else:
        return 1.0 / 0.7, '3x'

def measure_elixir_bar(elixir_crop_rgb: np.ndarray) -> float:
    eh, ew = elixir_crop_rgb.shape[:2]
    strip = elixir_crop_rgb[int(eh * 0.52):int(eh * 0.86), int(ew * 0.165):int(ew * 0.985), :]
    if strip.size == 0:
        return 0.0

    r = strip[:, :, 0].astype(np.int16)
    g = strip[:, :, 1].astype(np.int16)
    b = strip[:, :, 2].astype(np.int16)

    mag_mask = (r > 150) & (b > 130) & (g < 110) & ((r - g) > 50)
    col_ratio = mag_mask.mean(axis=0)
    active_cols = np.where(col_ratio > 0.25)[0]
    if len(active_cols) == 0:
        return 0.0

    rightmost = float(np.percentile(active_cols, 97))
    fill_fraction = np.clip((rightmost + 1.0) / strip.shape[1], 0.0, 1.0)
    return round(fill_fraction * 10.0, 2)

def is_chat_menu_occluding(slot_crops_rgb: list[np.ndarray]) -> bool:
    uniform_count = 0
    for s_rgb in slot_crops_rgb:
        sh, sw = s_rgb.shape[:2]
        center = s_rgb[int(sh * 0.25):int(sh * 0.75), int(sw * 0.15):int(sw * 0.85)]
        r_m, g_m, b_m = center[:, :, 0].mean(), center[:, :, 1].mean(), center[:, :, 2].mean()
        std_val = center.std()
        if (b_m > r_m + 35 and std_val < 38) or (r_m > 210 and g_m > 210 and b_m > 210 and std_val < 35):
            uniform_count += 1
    return uniform_count >= 3

def compute_slot_rotation_score(slot_pre_rgb: np.ndarray, slot_mid_rgb: np.ndarray, slot_post_rgb: np.ndarray) -> float:
    W_S, H_S = 36, 48
    pre_s  = cv2.resize(slot_pre_rgb,  (W_S, H_S), interpolation=cv2.INTER_AREA)
    mid_s  = cv2.resize(slot_mid_rgb,  (W_S, H_S), interpolation=cv2.INTER_AREA)
    post_s = cv2.resize(slot_post_rgb, (W_S, H_S), interpolation=cv2.INTER_AREA)

    pre_lab  = cv2.cvtColor(pre_s,  cv2.COLOR_RGB2LAB).astype(np.float32)
    mid_lab  = cv2.cvtColor(mid_s,  cv2.COLOR_RGB2LAB).astype(np.float32)
    post_lab = cv2.cvtColor(post_s, cv2.COLOR_RGB2LAB).astype(np.float32)

    patch_pre = pre_lab[10:36, 5:31]
    min_post_diff = 999.0
    for y0 in range(2, 19):
        cand_post = post_lab[y0:y0+26, 5:31]
        diff = float(np.abs(patch_pre - cand_post).mean())
        if diff < min_post_diff:
            min_post_diff = diff

    mid_diff = float(np.abs(patch_pre - mid_lab[10:36, 5:31]).mean())
    return 0.75 * min_post_diff + 0.25 * mid_diff

def identify_card_in_slot(
    slot_crop_rgb: np.ndarray,
    estimated_elixir_cost: float = None,
    elixir_before: float = None,
    allowed_base_keys: set[str] = None,
    top_k: int = 5
) -> list[dict]:
    sh, sw = slot_crop_rgb.shape[:2]
    candidate_patches = []
    for y_start_frac in (0.10, 0.05, 0.01):
        y0 = int(sh * (y_start_frac + 0.14))
        y1 = int(sh * (y_start_frac + 0.70))
        x0 = int(sw * 0.12)
        x1 = int(sw * 0.88)
        patch = slot_crop_rgb[max(0, y0):min(sh, y1), max(0, x0):min(sw, x1)]
        if patch.size == 0:
            continue
        patch_res = cv2.resize(patch, (TEMPLATE_W, TEMPLATE_H), interpolation=cv2.INTER_AREA)
        patch_lab = cv2.cvtColor(patch_res, cv2.COLOR_RGB2LAB).astype(np.float32)
        patch_gray = cv2.cvtColor(patch_res, cv2.COLOR_RGB2GRAY).astype(np.float32)
        patch_hsv = cv2.cvtColor(patch_res, cv2.COLOR_RGB2HSV)
        p_hist = cv2.calcHist([patch_hsv], [0, 1], None, [18, 16], [0, 180, 0, 256])
        cv2.normalize(p_hist, p_hist, alpha=1.0, norm_type=cv2.NORM_L1)
        candidate_patches.append((patch_lab, patch_gray, p_hist))

    max_affordable = (elixir_before + 0.85) if elixir_before is not None else 10.5
    scored = []

    for tmpl in card_templates:
        c_cost = tmpl['elixir']
        if c_cost > max_affordable:
            continue

        best_visual_sim = -1.0
        for p_lab, p_gray, p_hist in candidate_patches:
            ncc = float(cv2.matchTemplate(p_gray, tmpl['art_gray'], cv2.TM_CCOEFF_NORMED)[0, 0])
            lab_diff = float(np.abs(p_lab - tmpl['art_lab']).mean()) / 255.0
            lab_sim = max(0.0, 1.0 - lab_diff * 2.2)
            hist_sim = float(cv2.compareHist(p_hist, tmpl['hsv_hist'], cv2.HISTCMP_INTERSECT))

            vis_score = 0.45 * ncc + 0.35 * lab_sim + 0.20 * hist_sim
            if vis_score > best_visual_sim:
                best_visual_sim = vis_score

        elixir_bonus = 0.0
        if estimated_elixir_cost is not None and c_cost > 0:
            cost_diff = abs(c_cost - estimated_elixir_cost)
            if cost_diff <= 0.75:
                elixir_bonus = 0.10
            elif cost_diff <= 1.4:
                elixir_bonus = 0.04
            elif cost_diff >= 2.1:
                elixir_bonus = -0.15

        deck_bonus = 0.08 if (allowed_base_keys and tmpl['base_key'] in allowed_base_keys) else 0.0
        total_score = best_visual_sim + elixir_bonus + deck_bonus
        scored.append((total_score, best_visual_sim, tmpl))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [
        {
            'key': tmpl['key'],
            'base_key': tmpl['base_key'],
            'name_es': tmpl['name_es'],
            'name_en': tmpl['name_en'],
            'elixir': tmpl['elixir'],
            'is_evolution': tmpl['is_evolution'],
            'score': round(float(tot_s), 4),
            'visual_score': round(float(vis_s), 4)
        }
        for tot_s, vis_s, tmpl in scored[:top_k]
    ]

print('✔ Algoritmos con restricción física de elixir e inferencia de mazo listos.')

✔ Algoritmos con restricción física de elixir e inferencia de mazo listos.


In [65]:
# =========================================================================
# ESCÁNER SECUENCIAL DE PARTIDA A 5 FPS + INFERENCIA DEL MAZO DE 8 CARTAS
# =========================================================================

SAMPLE_FPS = 5.0

def extract_match_events(cap: cv2.VideoCapture, match_info: dict, layout: dict) -> dict:
    start_sec = float(match_info['start_sec'])
    end_sec   = float(match_info['end_sec'])
    clock_offset_sec = get_battle_clock_offset(match_info)

    px = layout['boxes_pixel']
    ex, ey, ew, eh = px['elixir_bar_box']
    slot_boxes = px['card_slots']

    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step_frames = max(1, int(round(video_fps / SAMPLE_FPS)))

    cap.set(cv2.CAP_PROP_POS_MSEC, start_sec * 1000.0)
    frame_idx = 0
    timeline = []

    while True:
        ret = cap.grab()
        if not ret:
            break
        if frame_idx % step_frames == 0:
            t_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
            if t_sec > end_sec:
                break
            if t_sec >= start_sec:
                ok, frame_bgr = cap.retrieve()
                if ok and frame_bgr is not None:
                    elix_rgb = cv2.cvtColor(frame_bgr[ey:ey+eh, ex:ex+ew], cv2.COLOR_BGR2RGB)
                    slots_rgb = [
                        cv2.cvtColor(frame_bgr[sy:sy+sh, sx:sx+sw], cv2.COLOR_BGR2RGB)
                        for (sx, sy, sw, sh) in slot_boxes
                    ]

                    e_val = measure_elixir_bar(elix_rgb)
                    occluded = is_chat_menu_occluding(slots_rgb)
                    battle_elapsed = (t_sec - start_sec) + clock_offset_sec

                    timeline.append({
                        't_sec': round(t_sec, 2),
                        'battle_sec': round(battle_elapsed, 2),
                        'elixir': e_val,
                        'occluded': occluded,
                        'slots_rgb': slots_rgb
                    })
        frame_idx += 1

    n_samples = len(timeline)
    if n_samples < 12:
        return {'timeline': [], 'events': [], 'detected_deck': [], 'clock_offset_sec': clock_offset_sec}

    raw_elixir = np.array([m['elixir'] for m in timeline], dtype=np.float32)
    smooth_elixir = np.copy(raw_elixir)
    for i in range(1, n_samples - 1):
        smooth_elixir[i] = np.median(raw_elixir[i-1:i+2])

    candidate_events = []
    for i in range(4, n_samples - 6):
        if timeline[i]['occluded'] or timeline[i-4]['occluded'] or timeline[i+5]['occluded']:
            continue

        t_before = timeline[i - 2]['t_sec']
        t_after  = timeline[i + 2]['t_sec']
        dt_win   = max(0.2, t_after - t_before)

        e_before = float(np.max(smooth_elixir[i-3:i+1]))
        e_after  = float(np.min(smooth_elixir[i:i+3]))
        regen_rate, phase = get_elixir_regen_rate(timeline[i]['battle_sec'])

        estimated_spend = (e_before - e_after) + (regen_rate * dt_win * 0.65)
        if estimated_spend < 0.90:
            continue

        slot_rot_scores = [
            compute_slot_rotation_score(
                timeline[i - 4]['slots_rgb'][s],
                timeline[i + 1]['slots_rgb'][s],
                timeline[i + 5]['slots_rgb'][s]
            )
            for s in range(4)
        ]
        best_slot = int(np.argmax(slot_rot_scores))
        max_rot = slot_rot_scores[best_slot]
        second_rot = sorted(slot_rot_scores, reverse=True)[1]

        if max_rot >= 12.0 and max_rot > second_rot * 1.30:
            ev_strength = estimated_spend * max_rot
            if candidate_events and (i - candidate_events[-1]['idx'] <= 5):
                if ev_strength > candidate_events[-1]['event_strength']:
                    candidate_events[-1] = {
                        'idx': i,
                        'slot': best_slot,
                        'phase': phase,
                        'estimated_spend': round(estimated_spend, 2),
                        'elixir_before': round(e_before, 2),
                        'elixir_after': round(e_after, 2),
                        'slot_rot_score': round(max_rot, 1),
                        'event_strength': ev_strength
                    }
            else:
                candidate_events.append({
                    'idx': i,
                    'slot': best_slot,
                    'phase': phase,
                    'estimated_spend': round(estimated_spend, 2),
                    'elixir_before': round(e_before, 2),
                    'elixir_after': round(e_after, 2),
                    'slot_rot_score': round(max_rot, 1),
                    'event_strength': ev_strength
                })

    deck_votes = defaultdict(float)
    for ev in candidate_events:
        pre_idx = max(0, ev['idx'] - 4)
        slot_pre_rgb = timeline[pre_idx]['slots_rgb'][ev['slot']]
        cands = identify_card_in_slot(
            slot_pre_rgb,
            estimated_elixir_cost=ev['estimated_spend'],
            elixir_before=ev['elixir_before'],
            top_k=3
        )
        if cands:
            top1 = cands[0]
            deck_votes[top1['base_key']] += max(0.1, top1['score'])

    inferred_deck_keys = set(
        k for k, _ in sorted(deck_votes.items(), key=lambda item: item[1], reverse=True)[:8]
    )

    events_out = []
    deck_play_counts = defaultdict(int)

    for ev_num, ev in enumerate(candidate_events, 1):
        idx = ev['idx']
        slot_i = ev['slot']
        pre_idx  = max(0, idx - 4)
        post_idx = min(n_samples - 1, idx + 5)

        slot_pre_rgb  = timeline[pre_idx]['slots_rgb'][slot_i]
        slot_post_rgb = timeline[post_idx]['slots_rgb'][slot_i]

        top_matches = identify_card_in_slot(
            slot_pre_rgb,
            estimated_elixir_cost=ev['estimated_spend'],
            elixir_before=ev['elixir_before'],
            allowed_base_keys=inferred_deck_keys,
            top_k=5
        )
        best_card = top_matches[0]
        deck_play_counts[best_card['base_key']] += 1

        events_out.append({
            'event_index': ev_num,
            't_drop_sec': timeline[idx]['t_sec'],
            't_decision_sec': timeline[pre_idx]['t_sec'],
            'battle_sec': timeline[idx]['battle_sec'],
            'elixir_phase': ev['phase'],
            'hand_slot': slot_i,
            'elixir_before': ev['elixir_before'],
            'elixir_after': ev['elixir_after'],
            'estimated_spend': ev['estimated_spend'],
            'slot_rot_score': ev['slot_rot_score'],
            'card_key': best_card['key'],
            'card_base_key': best_card['base_key'],
            'card_name_es': best_card['name_es'],
            'card_elixir': best_card['elixir'],
            'is_evolution': best_card['is_evolution'],
            'confidence': best_card['score'],
            'top_candidates': top_matches[:3],
            '_slot_pre_rgb': slot_pre_rgb,
            '_slot_post_rgb': slot_post_rgb
        })

    detected_deck_summary = [
        {
            'base_key': bk,
            'name_es': cards_dict.get(bk, {}).get('name_es', bk),
            'elixir': cards_dict.get(bk, {}).get('elixir', 0),
            'times_played': count
        }
        for bk, count in sorted(deck_play_counts.items(), key=lambda x: (-x[1], x[0]))[:8]
    ]

    light_timeline = [
        {'t_sec': m['t_sec'], 'battle_sec': m['battle_sec'], 'elixir': float(smooth_elixir[k]), 'occluded': m['occluded']}
        for k, m in enumerate(timeline)
    ]
    return {
        'clock_offset_sec': clock_offset_sec,
        'detected_deck': detected_deck_summary,
        'timeline': light_timeline,
        'events': events_out
    }

print('✔ Escáner secuencial de partida a 5 FPS listo.')

✔ Escáner secuencial de partida a 5 FPS listo.


In [ ]:
# =========================================================================
# BUCLE DE EJECUCIÓN CON CHECKPOINTING ATÓMICO Y CONSOLIDACIÓN EN R2
# =========================================================================

SKIP_ALREADY_PROCESSED = True
DELETE_VIDEO_AFTER = len(selected_videos) > 2

def download_video_from_cdn(video_id: str) -> Path:
    target = VIDEOS_DIR / f'{video_id}.mp4'
    if target.exists() and target.stat().st_size > 1024 * 1024:
        return target
    url = f'{CDN_BASE}/videos/{video_id}.mp4'
    t0 = time.time()
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(target, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    mb = target.stat().st_size / (1024 * 1024)
    print(f'  ⬇ Video {video_id}.mp4 descargado ({mb:.1f} MB en {time.time() - t0:.1f}s)', flush=True)
    return target

def save_events_checkpoint(video_id: str, payload: dict):
    clean_payload = json.loads(json.dumps(payload, default=lambda o: None))
    for m in clean_payload.get('matches', []):
        for ev in m.get('events', []):
            ev.pop('_slot_pre_rgb', None)
            ev.pop('_slot_post_rgb', None)

    local_file = EVENTS_DIR / f'{video_id}.json'
    local_file.write_text(json.dumps(clean_payload, indent=2, ensure_ascii=False), encoding='utf-8')

    try:
        s3_key = f'metadata/events/{video_id}.json'
        s3_client.put_object(
            Bucket=R2_BUCKET,
            Key=s3_key,
            Body=json.dumps(clean_payload, indent=2, ensure_ascii=False).encode('utf-8'),
            ContentType='application/json'
        )
        print(f'  ☁ [{video_id}] Checkpoint guardado en R2: {s3_key}', flush=True)
    except Exception as e:
        print(f'  ⚠ Error subiendo checkpoint de {video_id} a R2: {e}', flush=True)

def consolidate_events_manifest() -> dict:
    all_events = {}
    for f in EVENTS_DIR.glob('*.json'):
        try:
            d = json.loads(f.read_text(encoding='utf-8'))
            all_events[d['video_id']] = d
        except Exception:
            pass
    try:
        res = s3_client.list_objects_v2(Bucket=R2_BUCKET, Prefix='metadata/events/')
        for item in res.get('Contents', []):
            vid = Path(item['Key']).stem
            if vid and vid not in all_events:
                obj = s3_client.get_object(Bucket=R2_BUCKET, Key=item['Key'])
                all_events[vid] = json.loads(obj['Body'].read().decode('utf-8'))
    except Exception as e:
        print(f'⚠ Error listando eventos en R2: {e}')

    total_plays_all = sum(v.get('total_events', 0) for v in all_events.values())
    manifest = {
        'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'total_videos': len(all_events),
        'total_events': total_plays_all,
        'videos': all_events
    }
    (MANIFESTS_DIR / 'extracted_events.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    try:
        s3_client.put_object(
            Bucket=R2_BUCKET,
            Key='metadata/extracted_events.json',
            Body=json.dumps(manifest, indent=2, ensure_ascii=False).encode('utf-8'),
            ContentType='application/json'
        )
        print(f"\n☁ Manifiesto maestro consolidado en R2: metadata/extracted_events.json ({len(all_events)} videos, {total_plays_all} jugadas totales)", flush=True)
    except Exception as e:
        print(f'⚠ Error subiendo extracted_events.json: {e}')
    return manifest

session_results = {}

for idx, v_item in enumerate(selected_videos, 1):
    vid = v_item['id']
    if SKIP_ALREADY_PROCESSED and vid in already_extracted and len(selected_videos) > 1:
        print(f'[{idx}/{len(selected_videos)}] ⏩ {vid}: ya procesado en R2, saltando.', flush=True)
        continue

    print(f"\n[{idx}/{len(selected_videos)}] 🎯 Extrayendo jugadas de {vid} ({len(v_item['matches'])} partida(s))...", flush=True)
    t_vid = time.time()
    video_path = download_video_from_cdn(vid)
    cap = cv2.VideoCapture(str(video_path))

    matches_results = []
    total_plays = 0
    for m_info in v_item['matches']:
        t_m = time.time()
        res = extract_match_events(cap, m_info, v_item['layout'])
        evs = res['events']
        total_plays += len(evs)
        deck_str = ', '.join(f"{d['name_es']} (x{d['times_played']})" for d in res['detected_deck'])
        print(
            f"  • Partida #{m_info['match_index']} ({m_info['start_sec']}s–{m_info['end_sec']}s, "
            f"start_clock={m_info.get('start_clock')}): {len(evs)} jugadas en {time.time() - t_m:.1f}s\n"
            f"    Mazo inferido ({len(res['detected_deck'])} cartas): {deck_str}",
            flush=True
        )
        matches_results.append({
            'match_id': m_info['match_id'],
            'match_index': m_info['match_index'],
            'start_sec': m_info['start_sec'],
            'end_sec': m_info['end_sec'],
            'duration_sec': m_info['duration_sec'],
            'start_clock': m_info.get('start_clock'),
            'end_clock': m_info.get('end_clock'),
            'clock_offset_sec': res['clock_offset_sec'],
            'detected_deck': res['detected_deck'],
            'events_count': len(evs),
            'events': evs,
            'timeline': res['timeline']
        })

    cap.release()
    if DELETE_VIDEO_AFTER and video_path.exists():
        video_path.unlink(missing_ok=True)

    payload = {
        'video_id': vid,
        'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'total_matches': len(matches_results),
        'total_events': total_plays,
        'matches': matches_results
    }
    save_events_checkpoint(vid, payload)
    session_results[vid] = payload
    already_extracted.add(vid)
    print(f'✔ {vid} completado en {time.time() - t_vid:.1f}s — Total jugadas: {total_plays}', flush=True)

consolidate_events_manifest()

[1/41] ⏩ FsfSvTL6kIs: ya procesado en R2, saltando.
[2/41] ⏩ XTiXwJ8ffRM: ya procesado en R2, saltando.
[3/41] ⏩ qlB3ewZywzs: ya procesado en R2, saltando.
[4/41] ⏩ AMMLa-tcCNM: ya procesado en R2, saltando.
[5/41] ⏩ 4YR1phwfrJU: ya procesado en R2, saltando.
[6/41] ⏩ I8tqmODo5YQ: ya procesado en R2, saltando.
[7/41] ⏩ pRvDwdcMaHs: ya procesado en R2, saltando.
[8/41] ⏩ tNt3uzi-lUo: ya procesado en R2, saltando.
[9/41] ⏩ 6Kyn3I_6YcE: ya procesado en R2, saltando.
[10/41] ⏩ 8GhKcJQhlDs: ya procesado en R2, saltando.
[11/41] ⏩ DEgMhEkeL9o: ya procesado en R2, saltando.
[12/41] ⏩ MmsVS9PHrk4: ya procesado en R2, saltando.
[13/41] ⏩ QnFw3kiwOqo: ya procesado en R2, saltando.
[14/41] ⏩ kKf43ehxl58: ya procesado en R2, saltando.
[15/41] ⏩ 2Q9wFQfkVu4: ya procesado en R2, saltando.
[16/41] ⏩ ghfnrYrG_Sk: ya procesado en R2, saltando.
[17/41] ⏩ oVrQIR5w_t4: ya procesado en R2, saltando.
[18/41] ⏩ sTlfwVbOfdE: ya procesado en R2, saltando.
[19/41] ⏩ 4Tt7nuuy-Bg: ya procesado en R2, saltando.
[2

KeyboardInterrupt: 

In [ ]:
# =========================================================================
# VISUALIZACIÓN 1: MAZO DETECTADO Y CURVA DE ELIXIR (1ra PARTIDA DE CADA VIDEO)
# - Funciona tanto con videos recién procesados como con videos ya guardados en R2.
# - Grafica como mínimo 1 video (la primera partida de cada video en selected_videos).
# =========================================================================
PLOT_VIDEO_FILTER = 'all'  # 'all' = todos los de selected_videos, o un entero (ej. 5)

extracted_ids_for_plot = get_processed_event_ids()
pool_for_plot = [v for v in selected_videos if v['id'] in extracted_ids_for_plot or v['id'] in session_results]
if not pool_for_plot and available_videos:
    # Fallback de seguridad: al menos mostrar 1 si hay alguno en R2
    pool_for_plot = [v for v in available_videos if v['id'] in extracted_ids_for_plot][:1]

plot_targets = select_videos(pool_for_plot, PLOT_VIDEO_FILTER)

if not plot_targets:
    print('No hay videos procesados todavía para graficar la curva de elixir.')
else:
    for item in plot_targets:
        vid = item['id']
        v_payload = session_results[vid] if vid in session_results else load_r2_json(f'metadata/events/{vid}.json', EVENTS_DIR / f'{vid}.json')
        if not v_payload.get('matches'):
            continue

        m_data = v_payload['matches'][0]  # Primera partida del video
        tl = m_data.get('timeline', [])
        evs = m_data.get('events', [])
        deck = m_data.get('detected_deck', [])

        if deck:
            fig_d, axes_d = plt.subplots(1, 8, figsize=(16, 2.3), facecolor='#0f172a')
            fig_d.suptitle(
                f"Mazo de 8 Cartas Inferido — Video {vid} (Partida #1: {len(evs)} jugadas)",
                color='white', fontsize=12.5, fontweight='bold', y=1.02
            )
            for i in range(8):
                ax_d = axes_d[i]
                ax_d.set_facecolor('#1e293b')
                ax_d.axis('off')
                if i < len(deck):
                    d_card = deck[i]
                    bk = d_card['base_key']
                    if bk in card_templates_by_key:
                        ax_d.imshow(card_templates_by_key[bk]['full_rgb'])
                    ax_d.set_title(
                        f"{d_card['name_es']}\n({d_card['elixir']}💧 • {d_card['times_played']}x)",
                        color='#e2e8f0', fontsize=8.5, pad=3
                    )
            plt.tight_layout()
            plt.show()

        if tl:
            ts = [p['t_sec'] for p in tl]
            es = [p['elixir'] for p in tl]
            t0_match = m_data['start_sec']
            offset_s = m_data.get('clock_offset_sec', 4.0)
            t_2x_start = t0_match + (120.0 - offset_s)
            t_3x_start = t0_match + (240.0 - offset_s)

            fig, ax = plt.subplots(figsize=(16, 4.0), facecolor='#111827')
            ax.set_facecolor('#1f2937')

            ax.axvspan(t0_match, min(m_data['end_sec'], t_2x_start), color='#1e3a8a', alpha=0.22, label='Elixir 1x (2.8s/gota)')
            if m_data['end_sec'] > t_2x_start:
                ax.axvspan(t_2x_start, min(m_data['end_sec'], t_3x_start), color='#7e22ce', alpha=0.22, label='Elixir 2x (1.4s/gota)')
            if m_data['end_sec'] > t_3x_start:
                ax.axvspan(t_3x_start, m_data['end_sec'], color='#be123c', alpha=0.25, label='Elixir 3x (0.7s/gota)')

            ax.plot(ts, es, color='#f472b6', linewidth=2.0, label='Elixir en barra (0-10)')

            for idx_ev, ev in enumerate(evs):
                tx = ev['t_drop_sec']
                eb = ev['elixir_before']
                ax.axvline(tx, color='#34d399', linestyle=':', alpha=0.65, linewidth=1.0)
                ax.scatter([tx], [eb], color='#10b981', edgecolor='white', s=45, zorder=5)
                y_lbl = min(10.6, eb + (0.45 if idx_ev % 2 == 0 else 1.15))
                ax.text(
                    tx, y_lbl, f"#{ev['event_index']}",
                    color='#a7f3d0', fontsize=8.0, fontweight='bold', ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='#064e3b', edgecolor='#10b981', alpha=0.85)
                )

            ax.set_ylim(-0.3, 11.6)
            ax.set_xlim(m_data['start_sec'], m_data['end_sec'])
            ax.set_title(
                f"Curva de Elixir a lo largo de la Partida #1 — Video {vid} ({len(evs)} jugadas detectadas)",
                color='white', fontsize=12.5, fontweight='bold'
            )
            ax.set_xlabel('Segundo del video (s)', color='#d1d5db')
            ax.set_ylabel('Elixir (0 - 10)', color='#d1d5db')
            ax.tick_params(colors='#d1d5db')
            ax.legend(loc='upper right', facecolor='#111827', edgecolor='#374151', labelcolor='white', fontsize=9)
            plt.tight_layout()
            plt.show()

In [ ]:
# =========================================================================
# VISUALIZACIÓN 2: GALERÍA ESPACIADA DE JUGADAS
# =========================================================================
if session_results:
    sample_vid = next(iter(session_results.keys()))
    m_data = session_results[sample_vid]['matches'][0]
    evs = [e for e in m_data['events'] if '_slot_pre_rgb' in e]

    if evs:
        cols = 5
        rows = int(np.ceil(len(evs) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.4, rows * 3.1), facecolor='#0f172a')
        axes_flat = np.array(axes).flatten() if isinstance(axes, np.ndarray) else [axes]

        for idx, ax in enumerate(axes_flat):
            ax.set_facecolor('#1e293b')
            ax.axis('off')
            if idx < len(evs):
                ev = evs[idx]
                pre_img = cv2.resize(ev['_slot_pre_rgb'], (105, 130))
                cat_img = cv2.resize(card_templates_by_key[ev['card_key']]['full_rgb'], (105, 130))
                sep = np.full((130, 8, 3), 15, dtype=np.uint8)
                pair_img = np.hstack([pre_img, sep, cat_img])

                ax.imshow(pair_img)
                evo_tag = ' [EVO]' if ev['is_evolution'] else ''
                mmss = fmt_mmss(ev['t_drop_sec'])

                ax.set_title(
                    f"#{ev['event_index']:02d} • {mmss} ({ev['t_drop_sec']}s, {ev['elixir_phase']}) • Slot {ev['hand_slot']+1}\n"
                    f"{ev['card_name_es']}{evo_tag} ({ev['card_elixir']}💧 | {ev['elixir_before']}→{ev['elixir_after']})",
                    color='#f8fafc', fontsize=9.0, fontweight='bold', pad=6
                )

        plt.suptitle(
            f'Galería de Jugadas Detectadas ({sample_vid}): [Izquierda: Slot del Video en t-0.8s]  vs  [Derecha: Carta Catálogo R2]',
            color='white', fontsize=14, fontweight='bold', y=1.01
        )
        plt.subplots_adjust(hspace=0.42, wspace=0.18)
        plt.show()

In [ ]:
# =========================================================================
# VISUALIZACIÓN 3: GENERADOR DE VIDEOS DE DEBUG BAJO DEMANDA (SIN REPROCESAR)
# - Lee directamente los JSON ya guardados en R2 (metadata/events/<vid>.json).
# - No necesitas volver a correr la celda de extracción para generar videos
#   nuevos más adelante.
# - Puedes elegir qué videos (DEBUG_VIDEO_FILTER) y si quieres solo la 1ra
#   partida de cada video (DEBUG_MATCH_FILTER = 1) o todas ('all').
# =========================================================================

DEBUG_VIDEO_FILTER = 'all'      # Ej: 2, 'Qy18H5V-nbE', ['Qy18H5V-nbE', 'soLMMwrsFIo'], o 'all'
DEBUG_MATCH_FILTER = 1      # 1 = solo primera partida de cada video | 'all' = todas las partidas del video
SKIP_EXISTING_DEBUG = False # Si True, si el video debug ya existe en R2 solo muestra el reproductor sin re-renderizar

def get_existing_r2_debug_videos() -> set[str]:
    existing = set()
    try:
        res = s3_client.list_objects_v2(Bucket=R2_BUCKET, Prefix='debug_videos/actions/')
        for item in res.get('Contents', []):
            existing.add(Path(item['Key']).name)
    except Exception:
        pass
    return existing

def generate_verification_video(
    video_id: str,
    v_payload: dict,
    match_idx_in_list: int = 0,
    out_fps: int = 10,
    existing_debug_files: set[str] = None
) -> str:
    m_data = v_payload['matches'][match_idx_in_list]
    m_num  = m_data.get('match_index', match_idx_in_list + 1)
    out_name = f'{video_id}_m{m_num}_verify.mp4'
    r2_debug_key = f'debug_videos/actions/{out_name}'

    if SKIP_EXISTING_DEBUG and existing_debug_files and out_name in existing_debug_files:
        cdn_video_url = f'{CDN_BASE}/{r2_debug_key}'
        print(f'⏩ Video debug {out_name} ya existe en R2: {cdn_video_url}', flush=True)
        return cdn_video_url

    lay = layouts_dict[video_id]
    px = lay['boxes_pixel']

    start_sec = float(m_data['start_sec'])
    end_sec   = float(m_data['end_sec'])
    events    = m_data['events']
    tl        = m_data['timeline']

    tl_times  = np.array([p['t_sec'] for p in tl], dtype=np.float32)
    tl_elixir = np.array([p['elixir'] for p in tl], dtype=np.float32)

    video_path = download_video_from_cdn(video_id)
    cap = cv2.VideoCapture(str(video_path))
    orig_w, orig_h = lay['resolution']
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step_f = max(1, int(round(src_fps / out_fps)))

    VIEW_H = 640
    scale = VIEW_H / float(orig_h)
    GAME_W = int(round(orig_w * scale))
    if GAME_W % 2 == 1:
        GAME_W += 1
    PANEL_W = 300
    CANVAS_W = GAME_W + PANEL_W

    out_mp4 = DEBUG_VIDEOS_DIR / out_name
    ffmpeg_cmd = [
        'ffmpeg', '-y',
        '-f', 'rawvideo',
        '-vcodec', 'rawvideo',
        '-s', f'{CANVAS_W}x{VIEW_H}',
        '-pix_fmt', 'rgb24',
        '-r', str(out_fps),
        '-i', '-',
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        '-preset', 'veryfast',
        '-crf', '25',
        '-movflags', '+faststart',
        str(out_mp4)
    ]

    print(f'🎬 Renderizando {out_name} ({start_sec}s–{end_sec}s a {out_fps} FPS)...', flush=True)
    t0 = time.time()
    proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    cap.set(cv2.CAP_PROP_POS_MSEC, start_sec * 1000.0)
    f_idx = 0

    ex, ey, ew, eh = [int(v * scale) for v in px['elixir_bar_box']]
    scaled_slots = [[int(v * scale) for v in s] for s in px['card_slots']]

    while True:
        ret = cap.grab()
        if not ret:
            break
        if f_idx % step_f == 0:
            t_sec = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
            if t_sec > end_sec:
                break
            if t_sec >= start_sec:
                ok, frame_bgr = cap.retrieve()
                if ok and frame_bgr is not None:
                    game_rgb = cv2.cvtColor(cv2.resize(frame_bgr, (GAME_W, VIEW_H)), cv2.COLOR_BGR2RGB)
                    canvas = np.zeros((VIEW_H, CANVAS_W, 3), dtype=np.uint8)
                    canvas[:, :GAME_W] = game_rgb
                    canvas[:, GAME_W:] = (17, 24, 39)

                    cur_elix = float(tl_elixir[np.argmin(np.abs(tl_times - t_sec))]) if len(tl_times) else 0.0
                    battle_s = (t_sec - start_sec) + m_data.get('clock_offset_sec', 4.0)
                    _, cur_phase = get_elixir_regen_rate(battle_s)

                    active_ev = None
                    active_mode = None
                    last_triggered_ev = None
                    past_events = []

                    for ev in events:
                        td = ev['t_drop_sec']
                        if td <= t_sec + 0.1:
                            past_events.append(ev)
                            last_triggered_ev = ev
                        if (td - 0.85) <= t_sec < td:
                            active_ev = ev
                            active_mode = 'DECISION'
                            if '_slot_pre_rgb' not in ev or ev['_slot_pre_rgb'] is None:
                                s_box = px['card_slots'][ev['hand_slot']]
                                ev['_slot_pre_rgb'] = cv2.cvtColor(
                                    frame_bgr[s_box[1]:s_box[1]+s_box[3], s_box[0]:s_box[0]+s_box[2]],
                                    cv2.COLOR_BGR2RGB
                                )
                        elif td <= t_sec <= (td + 1.25):
                            active_ev = ev
                            active_mode = 'DROP'

                    cv2.rectangle(canvas, (ex, ey), (ex + ew, ey + eh), (255, 0, 255), 2)
                    for s_i, (sx, sy, sw, sh) in enumerate(scaled_slots):
                        if active_ev and active_ev['hand_slot'] == s_i:
                            if active_mode == 'DECISION':
                                cv2.rectangle(canvas, (sx-2, sy-2), (sx+sw+2, sy+sh+2), (250, 204, 21), 4)
                            else:
                                cv2.rectangle(canvas, (sx-3, sy-3), (sx+sw+3, sy+sh+3), (16, 185, 129), 4)
                        else:
                            cv2.rectangle(canvas, (sx, sy), (sx + sw, sy + sh), (0, 200, 120), 1)

                    if active_ev:
                        b_color = (180, 130, 0) if active_mode == 'DECISION' else (6, 95, 70)
                        cv2.rectangle(canvas, (8, VIEW_H - 215), (GAME_W - 8, VIEW_H - 175), b_color, -1)
                        cv2.rectangle(canvas, (8, VIEW_H - 215), (GAME_W - 8, VIEW_H - 175), (255, 255, 255), 1)

                    px0 = GAME_W + 16
                    bar_w = PANEL_W - 32
                    cv2.rectangle(canvas, (px0, 62), (px0 + bar_w, 84), (55, 65, 81), -1)
                    fill_w = int(bar_w * np.clip(cur_elix / 10.0, 0.0, 1.0))
                    cv2.rectangle(canvas, (px0, 62), (px0 + fill_w, 84), (236, 72, 153), -1)
                    cv2.rectangle(canvas, (px0, 62), (px0 + bar_w, 84), (244, 114, 182), 1)

                    if last_triggered_ev:
                        if '_slot_pre_rgb' in last_triggered_ev and last_triggered_ev['_slot_pre_rgb'] is not None:
                            pre_thumb = cv2.resize(last_triggered_ev['_slot_pre_rgb'], (105, 130))
                            canvas[145:145+130, px0:px0+105] = pre_thumb
                            cv2.rectangle(canvas, (px0, 145), (px0+105, 145+130), (148, 163, 184), 1)
                        ck = last_triggered_ev['card_key']
                        if ck in card_templates_by_key:
                            cat_thumb = cv2.resize(card_templates_by_key[ck]['full_rgb'], (105, 130))
                            canvas[145:145+130, px0+135:px0+240] = cat_thumb
                            cv2.rectangle(canvas, (px0+135, 145), (px0+240, 145+130), (52, 211, 153), 2)

                    pil_img = Image.fromarray(canvas)
                    draw = ImageDraw.Draw(pil_img)

                    draw.text((px0, 14), f"Video: {video_id} (P#{m_num}) | t={t_sec:5.1f}s ({fmt_mmss(t_sec)})", fill=(248, 250, 252))
                    draw.text((px0, 36), f"Elixir medido: {cur_elix:4.1f} / 10.0   [Fase {cur_phase}]", fill=(244, 114, 182))

                    if active_ev:
                        tag = 't-0.8s (FRAME ACTUAL VLM)' if active_mode == 'DECISION' else 'JUGADA EJECUTADA'
                        draw.text(
                            (16, VIEW_H - 208),
                            f"#{active_ev['event_index']} {tag}: {active_ev['card_name_es']} (Slot {active_ev['hand_slot']+1})",
                            fill=(255, 255, 255)
                        )

                    draw.text((px0, 102), "ULTIMA JUGADA DETECTADA:", fill=(148, 163, 184))
                    if last_triggered_ev:
                        draw.text((px0, 124), "[Slot t-0.8s]          [Catalogo R2]", fill=(203, 213, 225))
                        draw.text(
                            (px0, 286),
                            f"#{last_triggered_ev['event_index']:02d} {last_triggered_ev['card_name_es']} ({last_triggered_ev['card_elixir']} elixir)",
                            fill=(52, 211, 153)
                        )
                        draw.text(
                            (px0, 306),
                            f"Slot {last_triggered_ev['hand_slot']+1}  |  Elixir: {last_triggered_ev['elixir_before']} -> {last_triggered_ev['elixir_after']} (-{last_triggered_ev['estimated_spend']})",
                            fill=(226, 232, 240)
                        )
                    else:
                        draw.text((px0, 180), "Esperando primera jugada...", fill=(100, 116, 139))

                    draw.text((px0, 345), f"HISTORIAL EN VIVO ({len(past_events)}/{len(events)}):", fill=(148, 163, 184))
                    for row_i, pev in enumerate(reversed(past_events[-11:])):
                        y_txt = 370 + row_i * 22
                        is_now = (active_ev and active_ev['event_index'] == pev['event_index'])
                        c_txt = (52, 211, 153) if is_now else (203, 213, 225)
                        draw.text(
                            (px0, y_txt),
                            f"#{pev['event_index']:02d} [{fmt_mmss(pev['t_drop_sec'])}] S{pev['hand_slot']+1}: {pev['card_name_es']} (-{pev['card_elixir']})",
                            fill=c_txt
                        )

                    proc.stdin.write(np.array(pil_img, dtype=np.uint8).tobytes())
        f_idx += 1

    cap.release()
    proc.stdin.close()
    proc.wait()

    mb = out_mp4.stat().st_size / (1024 * 1024)
    print(f'✔ Video renderizado ({mb:.1f} MB en {time.time() - t0:.1f}s): {out_mp4}', flush=True)

    s3_client.upload_file(
        str(out_mp4), R2_BUCKET, r2_debug_key,
        ExtraArgs={'ContentType': 'video/mp4'}
    )
    cdn_video_url = f'{CDN_BASE}/{r2_debug_key}?v={int(time.time())}'
    print(f'☁ Disponible en CDN: {cdn_video_url}', flush=True)
    return cdn_video_url

# Construimos el pool sobre TODOS los videos ya extraídos en R2 (no requiere haber corrido la celda 5 hoy)
extracted_ids_now = get_processed_event_ids()
pool_for_debug = [v for v in selected_videos if v['id'] in extracted_ids_now]
if not DEBUG_VIDEO_FILTER or DEBUG_VIDEO_FILTER in ([], 0, False):
    print('⏩ Generación de video de verificación desactivada (DEBUG_VIDEO_FILTER está vacío).')
    debug_targets = []
else:
    debug_targets = select_videos(pool_for_debug, DEBUG_VIDEO_FILTER)
existing_debug = get_existing_r2_debug_videos()

for item in debug_targets:
    vid = item['id']
    v_payload = session_results[vid] if vid in session_results else load_r2_json(f'metadata/events/{vid}.json', EVENTS_DIR / f'{vid}.json')
    matches_list = v_payload.get('matches', [])
    if DEBUG_MATCH_FILTER == 'all':
        m_indices = list(range(len(matches_list)))
    elif isinstance(DEBUG_MATCH_FILTER, int):
        m_indices = list(range(min(DEBUG_MATCH_FILTER, len(matches_list))))
    else:
        m_indices = [0] if matches_list else []

    for m_idx in m_indices:
        cdn_url = generate_verification_video(vid, v_payload, match_idx_in_list=m_idx, out_fps=10, existing_debug_files=existing_debug)
        m_num = matches_list[m_idx].get('match_index', m_idx + 1)
        display(HTML(f"""
        <div style="background:#0f172a; padding:14px; border-radius:10px; display:inline-block; color:#f8fafc; font-family:sans-serif; margin-bottom:12px;">
          <div style="margin-bottom:8px; font-size:14px;">
            <b>🎥 Video de Contraste ({vid} — Partida #{m_num})</b> — 
            <a href="{cdn_url}" target="_blank" style="color:#38bdf8;">Abrir en pestaña nueva (CDN)</a>
          </div>
          <video width="660" controls preload="metadata" style="border-radius:6px; border:1px solid #334155;">
            <source src="{cdn_url}" type="video/mp4">
          </video>
        </div>
        """))

    # Si estamos renderizando muchos videos, limpiamos el .mp4 crudo al terminar cada video
    raw_mp4 = VIDEOS_DIR / f'{vid}.mp4'
    if len(debug_targets) > 2 and raw_mp4.exists():
        raw_mp4.unlink(missing_ok=True)